In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Proyecto Final - Anemia Infantil en Ayacucho
# MAGIC 
# MAGIC **Curso:** Sistemas de Gestión de Datos  
# MAGIC **Proyecto:** Lakehouse Analítico para el Monitoreo de Anemia Infantil en Provincias de Ayacucho 2023-2025  
# MAGIC 
# MAGIC En este notebook se configuran las rutas, catálogo, schemas y archivos fuente del proyecto.

In [0]:
# Configuración general del proyecto

catalog_name = "anemia_ayacucho"

schema_landing = "landing"
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold = "gold"

volume_name = "fuentes"

path_base = f"/Volumes/{catalog_name}/{schema_landing}/{volume_name}/input"

print("Catálogo del proyecto:", catalog_name)
print("Ruta base de archivos:", path_base)

Catálogo del proyecto: anemia_ayacucho
Ruta base de archivos: /Volumes/anemia_ayacucho/landing/fuentes/input


In [0]:
# Validación del catálogo y schemas del proyecto

spark.sql(f"USE CATALOG {catalog_name}")

print("Schemas disponibles en el catálogo:")
display(spark.sql(f"SHOW SCHEMAS IN {catalog_name}"))

Schemas disponibles en el catálogo:


databaseName
bronze
default
gold
information_schema
landing
silver


In [0]:
# Validación de archivos cargados en el volumen

archivos = dbutils.fs.ls(path_base)

display(archivos)

path,name,size,modificationTime
dbfs:/Volumes/anemia_ayacucho/landing/fuentes/input/data_2023.csv,data_2023.csv,24631,1782454010000
dbfs:/Volumes/anemia_ayacucho/landing/fuentes/input/data_2024.csv,data_2024.csv,24598,1782454011000
dbfs:/Volumes/anemia_ayacucho/landing/fuentes/input/data_2025.csv,data_2025.csv,24514,1782454010000
dbfs:/Volumes/anemia_ayacucho/landing/fuentes/input/maestro_zonas.json,maestro_zonas.json,3056,1782454010000


In [0]:
# Lista de archivos esperados para el proyecto

archivos_esperados = [
    "data_2023.csv",
    "data_2024.csv",
    "data_2025.csv",
    "maestro_zonas.json"
]

archivos_encontrados = [archivo.name for archivo in dbutils.fs.ls(path_base)]

for archivo in archivos_esperados:
    if archivo in archivos_encontrados:
        print(f"OK - Archivo encontrado: {archivo}")
    else:
        print(f"FALTA - Archivo no encontrado: {archivo}")

OK - Archivo encontrado: data_2023.csv
OK - Archivo encontrado: data_2024.csv
OK - Archivo encontrado: data_2025.csv
OK - Archivo encontrado: maestro_zonas.json


In [0]:
# Lectura de prueba del archivo CSV 2023

df_2023_prueba = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .csv(f"{path_base}/data_2023.csv")
)

display(df_2023_prueba.limit(10))

UBIGEO,PROVINCIA,DISTRITO,N° EVALUADOS,N° NIÑOS CON ANEMIA,AÑO,ZONAS
1,Cangallo,Cangallo,21,6,2023,Zona Norte
1,Cangallo,Cangallo,21,6,2023,Zona Sur
1,Cangallo,Cangallo,21,6,2023,Zona Este
1,Cangallo,Cangallo,21,6,2023,Zona Oeste
1,Cangallo,Cangallo,21,6,2023,Zona Centro
1,Cangallo,Cangallo,21,6,2023,Zona Noreste
1,Cangallo,Cangallo,20,5,2023,Zona Noroeste
1,Cangallo,Cangallo,20,5,2023,Zona Sureste
1,Cangallo,Cangallo,20,5,2023,Zona Suroeste
2,Cangallo,Chuschi,26,11,2023,Zona Norte


In [0]:
# Conteo de registros del archivo 2023

total_2023 = df_2023_prueba.count()

print("Cantidad de registros en data_2023.csv:", total_2023)

Cantidad de registros en data_2023.csv: 549


In [0]:
# Lectura de prueba del archivo JSON de zonas

df_zonas_prueba = (
    spark.read
    .option("multiline", "true")
    .json(f"{path_base}/maestro_zonas.json")
)

display(df_zonas_prueba)

abreviatura,descripcion,id_zona,orden_visualizacion,orientacion,tipo_zona,usar_en_dashboard,zona
N,Clasificación territorial correspondiente al sector norte del ámbito analizado.,1,1,Norte,Cardinal,true,Zona Norte
S,Clasificación territorial correspondiente al sector sur del ámbito analizado.,2,2,Sur,Cardinal,true,Zona Sur
E,Clasificación territorial correspondiente al sector este del ámbito analizado.,3,3,Este,Cardinal,true,Zona Este
O,Clasificación territorial correspondiente al sector oeste del ámbito analizado.,4,4,Oeste,Cardinal,true,Zona Oeste
C,Clasificación territorial correspondiente al sector central del ámbito analizado.,5,5,Centro,Central,true,Zona Centro
NE,Clasificación territorial correspondiente al sector noreste del ámbito analizado.,6,6,Noreste,Intercardinal,true,Zona Noreste
NO,Clasificación territorial correspondiente al sector noroeste del ámbito analizado.,7,7,Noroeste,Intercardinal,true,Zona Noroeste
SE,Clasificación territorial correspondiente al sector sureste del ámbito analizado.,8,8,Sureste,Intercardinal,true,Zona Sureste
SO,Clasificación territorial correspondiente al sector suroeste del ámbito analizado.,9,9,Suroeste,Intercardinal,true,Zona Suroeste


In [0]:
# Resumen inicial de fuentes de datos

resumen_fuentes = [
    ("data_2023.csv", "CSV", df_2023_prueba.count(), "Datos de anemia infantil 2023"),
    ("data_2024.csv", "CSV", spark.read.option("header", "true").option("sep", ";").csv(f"{path_base}/data_2024.csv").count(), "Datos de anemia infantil 2024"),
    ("data_2025.csv", "CSV", spark.read.option("header", "true").option("sep", ";").csv(f"{path_base}/data_2025.csv").count(), "Datos de anemia infantil 2025"),
    ("maestro_zonas.json", "JSON", df_zonas_prueba.count(), "Tabla maestra de zonas")
]

columnas = ["archivo", "formato", "cantidad_registros", "descripcion"]

df_resumen_fuentes = spark.createDataFrame(resumen_fuentes, columnas)

display(df_resumen_fuentes)

archivo,formato,cantidad_registros,descripcion
data_2023.csv,CSV,549,Datos de anemia infantil 2023
data_2024.csv,CSV,549,Datos de anemia infantil 2024
data_2025.csv,CSV,549,Datos de anemia infantil 2025
maestro_zonas.json,JSON,9,Tabla maestra de zonas
